In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Display settings (so dataframes don't truncate)
pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)

m6a_url = "https://osdr.nasa.gov/geode-py/ws/studies/OSD-569/download?source=datamanager&file=GLDS-561_directm6Aseq_Direct_RNA_seq_m6A_Processed_Data.xlsx"

m6a = pd.read_excel(
    m6a_url,
    sheet_name='I4-FP1',
    skiprows=[0, 1, 2, 3, 4, 5, 6, 7],
    header=[0, 1],
    index_col=0
)

# Flatten multi-level columns
m6a.columns = [f"{str(a)}_{str(b)}".strip("_") for a, b in m6a.columns]

print(m6a.shape)
print(m6a.columns.tolist())
m6a.head()

(57318, 22)
['transcript_position', 'gene_ENSEMBL', 'gene_position', 'C001_L-92', 'C001_L-44', 'C001_L-3', 'C001_R+1', 'C002_L-92', 'C002_L-44', 'C002_L-3', 'C002_R+1', 'C003_L-92', 'C003_L-44', 'C003_L-3', 'C003_R+1', 'C004_L-92', 'C004_L-44', 'C004_L-3', 'C004_R+1', 'I4-FP1_methylKit p-value', 'I4-FP1_methylKit q-value', 'I4-FP1_methylKit methylation difference, %']


,transcript_position,gene_ENSEMBL,gene_position,C001_L-92,C001_L-44,C001_L-3,C001_R+1,C002_L-92,C002_L-44,C002_L-3,C002_R+1,C003_L-92,C003_L-44,C003_L-3,C003_R+1,C004_L-92,C004_L-44,C004_L-3,C004_R+1,I4-FP1_methylKit p-value,I4-FP1_methylKit q-value,"I4-FP1_methylKit methylation difference, %"
ENST00000000233.10,178,ENSG00000004059.11,203,0.092703,NaN,NaN,0.092232,NaN,NaN,NaN,NaN,NaN,NaN,0.102265,NaN,NaN,NaN,0.067738,0.114244,1.0,1.0,2.357883
ENST00000000233.10,195,ENSG00000004059.11,220,NaN,NaN,NaN,0.154156,NaN,NaN,NaN,NaN,NaN,NaN,0.077314,NaN,NaN,NaN,0.172628,0.047985,1.0,1.0,2.426250
ENST00000000233.10,250,ENSG00000004059.11,275,NaN,NaN,NaN,0.014407,NaN,NaN,NaN,NaN,NaN,NaN,0.005053,0.010802,NaN,NaN,0.012737,0.016062,1.0,1.0,6.959631
ENST00000000233.10,266,ENSG00000004059.11,291,0.485814,0.484443,NaN,0.397263,NaN,NaN,NaN,NaN,NaN,NaN,0.318527,0.142476,NaN,NaN,0.508841,0.292754,1.0,1.0,1.389322
ENST00000000233.10,302,ENSG00000004059.11,327,0.119683,0.145970,NaN,0.178793,NaN,NaN,NaN,NaN,NaN,NaN,0.140672,0.274632,0.160893,NaN,0.118315,0.152584,1.0,1.0,2.651987


In [5]:
for col in m6a.columns:
    print(col)

transcript_position
gene_ENSEMBL
gene_position
C001_L-92
C001_L-44
C001_L-3
C001_R+1
C002_L-92
C002_L-44
C002_L-3
C002_R+1
C003_L-92
C003_L-44
C003_L-3
C003_R+1
C004_L-92
C004_L-44
C004_L-3
C004_R+1
I4-FP1_methylKit p-value
I4-FP1_methylKit q-value
I4-FP1_methylKit methylation difference, %


In [9]:
# Filter significant m6A positions
m6a_filtered = m6a[
    (m6a['I4-FP1_methylKit q-value'] < 0.05) &
    (m6a['I4-FP1_methylKit methylation difference, %'].abs() > 10)
].copy()

print(f"Total positions: {len(m6a)}")
print(f"Significant positions: {len(m6a_filtered)}")
print(m6a_filtered[['gene_ENSEMBL', 'I4-FP1_methylKit q-value', 'I4-FP1_methylKit methylation difference, %']].head(10))



Total positions: 57318
Significant positions: 94
                          gene_ENSEMBL  I4-FP1_methylKit q-value  \
ENST00000037243.7    ENSG00000034713.8              9.269566e-07   
ENST00000215909.10  ENSG00000100097.12              8.200972e-06   
ENST00000217652.8   ENSG00000101608.13              1.066000e-06   
ENST00000221930.6   ENSG00000105329.11              3.655758e-06   
ENST00000228825.12  ENSG00000111229.16              1.238010e-10   
ENST00000229379.3    ENSG00000111775.3              1.246618e-04   
ENST00000229379.3    ENSG00000111775.3              3.975542e-05   
ENST00000235382.7    ENSG00000116741.8              3.220294e-12   
ENST00000237654.9   ENSG00000118816.11              6.521387e-06   
ENST00000237654.9   ENSG00000118816.11              1.344878e-09   

                    I4-FP1_methylKit methylation difference, %  
ENST00000037243.7                                    16.790933  
ENST00000215909.10                                  -14.147474  
ENST000

In [10]:
# Collapse to unique genes
m6a_genes = m6a_filtered['gene_ENSEMBL'].unique().tolist()
print(f"Unique genes with significant m6A: {len(m6a_genes)}")
print(m6a_genes[:10])

Unique genes with significant m6A: 66
['ENSG00000034713.8', 'ENSG00000100097.12', 'ENSG00000101608.13', 'ENSG00000105329.11', 'ENSG00000111229.16', 'ENSG00000111775.3', 'ENSG00000116741.8', 'ENSG00000118816.11', 'ENSG00000122862.5', 'ENSG00000126264.10']


In [12]:
degs = pd.read_csv('degs_FP1.csv', index_col=0)
print(degs.columns.tolist())
print(degs.shape)

['C001_L-92', 'C001_L-44', 'C001_L-3', 'C001_R+1', 'C002_L-92', 'C002_L-44', 'C002_L-3', 'C002_R+1', 'C003_L-92', 'C003_L-44', 'C003_L-3', 'C003_R+1', 'C004_L-92', 'C004_L-44', 'C004_L-3', 'C004_R+1', 'C001_L-92.1', 'C001_L-44.1', 'C001_L-3.1', 'C001_R+1.1', 'C002_L-92.1', 'C002_L-44.1', 'C002_L-3.1', 'C002_R+1.1', 'C003_L-92.1', 'C003_L-44.1', 'C003_L-3.1', 'C003_R+1.1', 'C004_L-92.1', 'C004_L-44.1', 'C004_L-3.1', 'C004_R+1.1', 'DESeq2_log2FC', 'DESeq2_p-value', 'DESeq2_adjusted p-value', 'pipeline-transcriptome-de_log2FC', 'pipeline-transcriptome-de_p-value', 'pipeline-transcriptome-de_adjusted p-value', 'ENSEMBL_clean', 'symbol', 'name', 'domain']
(61, 42)


In [13]:
logfc_col = 'pipeline-transcriptome-de_log2FC'
padj_col = 'pipeline-transcriptome-de_adjusted p-value'

# Strip version numbers from m6A genes
m6a_genes_clean = [g.split('.')[0] for g in m6a_genes]

# Overlap
overlap = set(m6a_genes_clean) & set(degs['ENSEMBL_clean'].tolist())
print(f"RNA-seq DEGs: {len(degs)}")
print(f"m6A genes: {len(m6a_genes_clean)}")
print(f"Overlap: {len(overlap)}")

# Get full info for overlapping genes
overlap_df = degs[degs['ENSEMBL_clean'].isin(overlap)][['symbol', logfc_col, padj_col, 'domain']]
print(overlap_df)

RNA-seq DEGs: 61
m6A genes: 66
Overlap: 2
                    symbol  pipeline-transcriptome-de_log2FC  \
ENSG00000198876.13  DCAF12                         -1.285779   
ENSG00000206177.7      HBM                         -1.468648   

                    pipeline-transcriptome-de_adjusted p-value  \
ENSG00000198876.13                                    0.016375   
ENSG00000206177.7                                     0.002992   

                                 domain  
ENSG00000198876.13  Protein_Degradation  
ENSG00000206177.7         Erythroid_RBC  
